# scMultiBench API quickstart (Colab)

This notebook runs **entirely in Colab**: it installs the `multibench` API,
explores the method registry, loads the shipped benchmark results, and draws
the standard figures. Run the first cell and let the kernel restart once -
then keep running from the next cell.

One scope note: *executing* an integration method needs that method's conda
environment (2-14 GB each) and reference data, which this notebook does not
provision - for that, follow a category tutorial on your own Linux machine.
Everything below runs here, now.

In [ ]:
# Colab ships without conda; this provisions it (the kernel restarts ONCE).
# On a machine that already has conda, this cell does nothing.
import importlib.util, shutil

def _has(mod):
    try:
        return importlib.util.find_spec(mod) is not None
    except ModuleNotFoundError:
        return False

if shutil.which("conda") or shutil.which("mamba"):
    print("conda available - nothing to do")
elif _has("google.colab"):
    !pip -q install condacolab
    import condacolab
    condacolab.install()   # restarts the kernel; afterwards, continue below
else:
    print("no conda found - install it first (mamba recommended); see the installation guide")

In [ ]:
import importlib.util, os
if importlib.util.find_spec("multibench") is None:
    !git clone --depth 1 https://github.com/DSichang/scMultiBench.git
    %cd scMultiBench
    !pip -q install -e .
elif os.path.isdir("/content/scMultiBench"):
    # reused Colab runtime: refresh the editable install to the latest code,
    # then drop the already-imported modules so the NEXT import sees it -
    # a live kernel never re-reads changed files on its own
    %cd /content/scMultiBench
    !git pull -q
    !pip -q install -e .
    import importlib, sys
    for _m in [m for m in list(sys.modules) if m == "multibench" or m.startswith("multibench.")]:
        del sys.modules[_m]
    importlib.invalidate_caches()
    print("multibench refreshed to the latest repository state")
else:
    print("multibench already installed")

In [ ]:
import multibench as mtb
import pandas as pd

print(len(mtb.list_methods()), "methods in the registry")
mtb.list_methods(category="vertical")

## Inspect a method

`method_info` returns everything the registry knows - language, environment,
availability, label needs, the upstream reference and the entry-point variants;
`find_methods` filters by what your data has (the filters hold per variant: a
method matches when one of its variants satisfies category, modalities,
`needs_labels` and `atac` together); `cite` emits the benchmark entry plus one
per method.

In [ ]:
info = mtb.method_info("Matilda")
{k: info[k] for k in ("id", "language", "env", "availability", "needs_labels", "notes", "repo_url", "reference", "supports")}

In [ ]:
mtb.find_methods(category="vertical", modalities=["rna", "adt"], needs_labels=False)

In [ ]:
print(mtb.cite(["Matilda"], fmt="text"))

## Load shipped results and draw the standard figures

The package ships the paper's tables (`source="published"`) and its own
re-run sweeps (`source="rerun"`), one long table per dataset - so the figures
reproduce here without running anything.

In [ ]:
long = mtb.load_results("vertical", dataset="D11", source="rerun")
fig = mtb.plot.bubble(long)
fig.set_dpi(110)
fig

In [ ]:
pair = mtb.load_results("diagonal", dataset=["D28", "D28s"], source="rerun")
mtb.plot.bubble(pair, aggregate="summary", require_complete=True,
                title="Summary of 2 diagonal datasets")

## Next steps

- the four integration tutorials (vertical / diagonal / mosaic / cross) in the
  docs walk the full pipeline, including your own dataset
- the hosted [interactive explorer](https://shiny.maths.usyd.edu.au/scMultiBench/)
  has the complete published rankings